# 02. Advanced Tool Interface 실습

모델이 도구를 정확히 고르고 인자를 채우게 하려면 무엇을 바꿔야 할까요? 이 노트북은 그 질문을 **서버가 실제로 내보내는 도구 정의**에서 시작합니다. 요구사항 인터뷰 MCP 서버(도구 7개: `start_interview`, `record_answer`, `get_session`, `freeze_seed`, `list_evidence`, `read_evidence`, `save_evidence`)에 `tools/list`를 보내 이름, 설명, 인자 형식, 출력 형식, 힌트를 읽고, 잘못된 호출이 어떤 모양으로 돌아오는지 직접 깨뜨려 봅니다. 마지막에는 같은 서버를 "풍부한 정의(rich)"와 "빈약한 정의(plain)"로 띄워 Claude가 실제로 다르게 행동하는지 작게 재어 봅니다.

이 과정은 하나의 사례로 이어집니다. 요구사항 인터뷰 서버는 세션 상태를 `draft` → `ready` → `frozen`으로 옮기고, Deep Agent(`create_deep_agent`, `ChatAnthropic` `claude-sonnet-5-5`, effort medium)가 그 서버를 도구로 씁니다. 모든 코드는 제공되고, 이 노트북은 점수를 매기지 않습니다. 목표는 "코드가 돌아간다"가 아니라 **왜 그 결과가 나왔는지 자기 말로 설명하는 것**입니다.

## 학습 목표

- `tools/list` 결과에서 모델이 도구를 고르고 인자를 채우는 데 쓰는 칸(이름, 설명, `inputSchema`, `outputSchema`, annotations)을 구분합니다.
- `Literal` 열거형이 JSON Schema의 `enum`으로 나가는 모습을 보고, 열거형이 입력 검증을 서버 앞단으로 옮긴다는 것을 확인합니다.
- ToolAnnotations가 **힌트이지 강제가 아님**을 코드로 확인하고, 강제는 서버 코드(권한 거부 미들웨어)가 한다는 것을 구분합니다.
- 업무 오류(`isError=False`, `success=false`)와 프로토콜 층 오류(`isError=True`)의 결과 모양이 다르다는 것을 실제 호출로 가르고, 이 차이가 scorer의 판정에 어떻게 이어지는지 읽습니다.
- 서버가 발급한 핸들(`session_id`)이 도구 인자로 되돌아오는 흐름을 보고, 발급하지 않은 값이 들어왔을 때를 확인합니다.
- 같은 서버의 rich와 plain 인터페이스를 Claude로 12회 실행해 `args_valid`, `tool_selection_ok`를 비교하고, Wilson 구간으로 "이 숫자가 얼마나 적게 증명하는지"를 읽습니다.

### 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 0 | 준비와 핵심 질문 | 5 |
| 1 | `tools/list` 읽기: 이름, 설명, enum, 스키마, outputSchema | 9 |
| 2 | ToolAnnotations: 힌트와 강제의 차이 | 5 |
| 3 | 서버가 발급하는 핸들 | 4 |
| 4 | 업무 오류와 `isError` | 7 |
| 5 | rich 대 plain: 오프라인 비교 | 4 |
| 6 | 수업 중 실측: 12회 실행 | 10 |
| 7 | 직접 해보기, 확인 질문, 정리 | 6 |

## 호출 규모

기본 실행에서 유료 셀은 **한 개**(6절의 12회 실행)입니다. Claude 호출은 약 60~110회(실행당 5~9회 추정, `docs/DESIGN.md` 8절), Jev 호출은 0회입니다. 두 변형 모두 게이트를 끈 서버이므로 Decision API를 쓰지 않습니다. 이 비교는 Langfuse에 연결하지 않고 `outputs/notebook/` 아래 로컬 `rows.jsonl`에만 기록합니다(`cap_lab/notebook.py:172`). 실행 시간은 이 노트북을 쓰는 동안 재지 않았습니다. 실행이 하나씩 순서대로 돌기 때문에 몇 분이 걸린다고만 예상하세요.

나머지 셀은 모델을 부르지 않습니다. `tools/list`와 `tools/call`을 직접 보내는 셀은 같은 컴퓨터에서 서버 프로세스를 띄울 뿐이라 비용이 없습니다. `직접 해보기`의 유료 셀 두 개는 `RUN_EXERCISE = False`, `RUN_OPTIONAL = False`로 잠겨 있습니다.

## 0. 준비와 핵심 질문

먼저 이 노트북에서 가장 많이 쓰는 도구를 정리합니다. 모두 `cap_lab/notebook.py`에 있습니다.

| 이름 | 하는 일 | 위치 |
|---|---|---|
| `connect_server` | 서버를 stdio로 띄워 `mcp.Client`를 줍니다. DB와 근거는 작업 폴더 아래에 씁니다 | `cap_lab/notebook.py:76` |
| `new_workdir` | `outputs/notebook/<이름>-<날짜-시각>/` 폴더를 만듭니다. 같은 이름을 같은 초에 두 번 쓰면 `FileExistsError`입니다 | `cap_lab/notebook.py:68` |
| `announce` | 유료 호출 직전에 규모를 한 줄로 알립니다 | `cap_lab/notebook.py:35` |
| `run_small_compare` | dev 사례를 변형 x 반복 x 사례로 실행하고 scorer로 점수를 매겨 행 목록을 줍니다 | `cap_lab/notebook.py:142` |
| `pass_rate_table` | 변형과 검사별 통과 수와 Wilson 95% 구간을 줍니다 | `cap_lab/notebook.py:180` |

핵심 질문은 하나입니다. **모델은 도구의 무엇을 보고 호출을 정할까요?** 사람은 서버 코드를 열어 보지만 모델은 그러지 못합니다. 모델이 받는 것은 서버가 `tools/list`에 담아 보낸 정의뿐입니다. 이 노트북의 Agent는 그 도구 정의를 `MCPAdapter.list_tools()`로 읽어 오고(`cap_lab/agent.py:114`), Skill을 읽은 뒤에야 모델에게 공개합니다(`cap_lab/agent.py:116`, 5번째 노트북 주제). 여기서는 공개 방식은 미뤄 두고, 서버가 **무엇을 내보내는지**부터 봅니다.

아래 셀은 키 상태를 확인합니다. 키 값은 출력하지 않고 `설정됨`/`미설정`만 보여 줍니다. 6절의 유료 셀에는 `ANTHROPIC_API_KEY`가 필요합니다. 비어 있다면 `.env.example`을 보고 `Day04_new/.env`를 직접 채우세요. 다른 셀은 키 없이 돌아갑니다.

In [ ]:
# [따라하기] 준비: 도우미를 가져오고 키 상태만 확인합니다. 모델 호출 0회.
import json
import re
from pathlib import Path

from cap_lab.experiment import Variant
from cap_lab.notebook import (
    announce, connect_server, key_table, new_workdir,
    pass_rate_table, run_small_compare, show_events, show_table,
)
from cap_lab.server.server import FaultPlan
from cap_lab.stats import wilson_ci

key_table()

## 1. `tools/list` 읽기

**문제.** 도구가 일곱 개일 때 모델이 `record_answer`를 부를지 `save_evidence`를 부를지는 무엇이 정할까요? 서버 코드의 함수 이름과 docstring, 인자의 타입 힌트가 모두 `tools/list` 응답으로 바뀌어 나갑니다. 이 절에서는 응답을 가공하지 않고 읽습니다.

**먼저 예측하세요.** 아래 셀을 실행하기 전에 세 가지를 적어 두세요. (1) 도구는 몇 개이고, 각 도구의 `description`은 서버의 어느 부분에서 왔을까요? (2) `record_answer`의 `field` 인자는 모델에게 어떤 모양으로 보일까요? 자유 문자열일까요, 정해진 값 목록일까요? (3) 서버 코드에서 `Literal["constraints", "acceptance_criteria"]`라고 쓴 부분은 JSON Schema에서 어떤 키가 될까요?

먼저 도우미 함수 하나를 정의합니다. 서버를 띄워 `tools/list`를 읽고, 도구 정의를 dict 목록으로 돌려줍니다. `model_dump(by_alias=True)`를 쓰는 이유는 프로토콜에 실린 이름(`inputSchema`, `readOnlyHint`)을 그대로 보려는 것입니다.

In [ ]:
async def fetch_tools(interface, label):
    """서버를 stdio로 띄워 tools/list를 읽고 도구 정의를 dict 목록으로 돌려준다. 모델 호출 0회.

    interface: "rich" 또는 "plain". 서버의 --interface 값이다.
    label: 작업 폴더 이름의 앞부분. 같은 초에 같은 이름을 두 번 쓰면 폴더가 겹쳐 실패한다.
    """
    workdir = new_workdir(label)
    async with connect_server(workdir, interface=interface) as client:
        listed = await client.list_tools()
    # exclude_none=True: 서버가 보내지 않은 칸(예: plain의 annotations)은 목록에서 빠진다.
    return [tool.model_dump(by_alias=True, exclude_none=True) for tool in listed.tools]

In [ ]:
# [따라하기] rich 서버의 tools/list를 읽고 도구마다 한 줄로 요약합니다. 모델 호출 0회.
rich_tools = await fetch_tools("rich", "list-rich")
print("도구 수:", len(rich_tools))

summary_rows = []
for tool in rich_tools:
    schema = tool["inputSchema"]
    summary_rows.append({
        "이름": tool["name"],
        "설명": tool["description"],
        "인자": ", ".join(schema.get("properties", {})),
        "필수 인자": ", ".join(schema.get("required", [])),
        "outputSchema 있음": "outputSchema" in tool,
        "annotations 있음": "annotations" in tool,
    })
show_table(summary_rows)

**관찰할 것.** 표의 `설명` 칸이 서버 함수의 docstring 첫 문장과 같은지 `cap_lab/server/server.py`에서 `start_interview`(253~255행)와 대조하세요. docstring이 곧 모델이 읽는 설명입니다. 설명에 "무엇을 하는가"만 있고 "언제 쓰지 않는가"는 없다는 것도 눈여겨보세요. 그 빈칸은 도구가 아니라 Skill(`skills/requirements-interview/SKILL.md`)이 채웁니다. 이 서버의 도구 설명은 일부러 한두 문장입니다.

도구 목록의 **순서**도 보세요. 응답 순서가 서버에 등록한 순서와 같고 요청마다 바뀌지 않아야 모델에게 보이는 프롬프트가 안정됩니다. `docs/DESIGN.md` 5절에 따르면 2026-07-28 명세는 `tools/list` 순서를 결정적으로 정했습니다. 이 노트북은 순서가 매번 같은지를 따로 반복 측정하지 않았습니다.

In [ ]:
# [따라하기] record_answer의 정의 전체를 JSON으로 읽습니다. 모델 호출 0회.
record_answer_tool = next(tool for tool in rich_tools if tool["name"] == "record_answer")
print(json.dumps(record_answer_tool, ensure_ascii=False, indent=2))

input_schema = record_answer_tool["inputSchema"]
print()
print("field 인자의 정의:", input_schema["properties"]["field"])
print("inputSchema에 '$schema' 키가 있는가:", "$schema" in input_schema)

`field` 인자에 `enum`이 있습니다. 서버의 `AnswerField = Literal["constraints", "acceptance_criteria"]`(`cap_lab/server/server.py:58`)가 JSON Schema의 `enum`으로 나간 것입니다. 이 한 줄이 두 가지 일을 합니다. 모델에게는 "이 두 값 중 하나"라는 선택지를 보여 주고, 서버에는 그 밖의 값을 도구 코드가 실행되기 전에 막는 검증을 줍니다. 4절에서 실제로 막히는 모습을 봅니다.

JSON Schema 방언은 어떨까요? 출력의 마지막 줄을 보세요. 이 서버는 `$schema` 키를 보내지 않았습니다. 그래도 방언은 정해져 있습니다. 설치된 `mcp_types`의 정의가 "`$schema`가 없으면 JSON Schema 2020-12"라고 적고 있습니다(`.venv/lib/python3.13/site-packages/mcp_types/_v2026_07_28/__init__.py:1052`, `.venv/lib/python3.13/site-packages` 아래). 즉 2020-12는 **기본값**이라 서버가 키로 선언하지 않아도 됩니다. 다른 방언을 쓰는 도구 서버를 만나면 그때는 `$schema`가 있을 것입니다. 그 경우를 이 서버에서 만들어 보지는 않았습니다.

또 하나 눈여겨볼 칸이 있습니다. `title`("Session Id", "Field")은 인자 이름에서 자동으로 만들어진 값이고, 이 서버는 인자마다 `description`을 따로 주지 않습니다. 인자의 의미는 도구 설명과 Skill이 전합니다. 어느 쪽이 맞는 선택인지는 7절의 실측이 일부를 말해 줍니다.

In [ ]:
# [바꿔보기] TOOL_NAME을 다른 도구 이름으로 바꿔 인자 정의를 비교합니다. 모델 호출 0회.
TOOL_NAME = "save_evidence"   # 예: "start_interview", "list_evidence", "read_evidence"

chosen_tool = next(tool for tool in rich_tools if tool["name"] == TOOL_NAME)
chosen_schema = chosen_tool["inputSchema"]
required = set(chosen_schema.get("required", []))

argument_rows = []
for argument_name, definition in chosen_schema.get("properties", {}).items():
    argument_rows.append({
        "인자": argument_name,
        "타입": definition.get("type"),
        "enum": definition.get("enum", ""),
        "필수": argument_name in required,
        "기본값": definition.get("default", "(없음)"),
    })
print(TOOL_NAME, "-", chosen_tool["description"])
show_table(argument_rows)

`save_evidence`는 인자가 네 개이고 마지막 `source_evidence_id`만 기본값이 있습니다. `list_evidence`의 `session_id`도 기본값 `""`입니다. **필수 여부는 모델이 인자를 채우는 방식에 영향을 줍니다.** 기본값이 있는 인자는 빼먹어도 호출이 성공하므로, 빼먹은 것이 실수인지 의도인지 서버가 구분하지 못합니다. Skill 6항이 "원본 근거 ID를 `source_evidence_id`에 넣는다"를 따로 적어 둔 이유입니다.

다음은 출력 쪽입니다. `outputSchema`는 서버가 "내 결과는 이런 모양이다"라고 알리는 계약입니다.

In [ ]:
# [따라하기] outputSchema를 읽고, 같은 도구의 실제 결과(structuredContent)와 나란히 놓습니다. 모델 호출 0회.
output_rows = []
for tool in rich_tools:
    output_schema = tool.get("outputSchema", {})
    output_rows.append({
        "이름": tool["name"],
        "타입": output_schema.get("type"),
        "additionalProperties": output_schema.get("additionalProperties"),
        "정해 둔 필드": ", ".join(output_schema.get("properties", {})) or "(없음)",
    })
show_table(output_rows)

workdir_output = new_workdir("output")
async with connect_server(workdir_output) as client:
    started = await client.call_tool("start_interview", {"goal": "팀 비품 대여 현황을 보여 주는 도구"})
print("start_interview가 실제로 돌려준 structuredContent의 키:", list(started.structured_content))

**관찰할 것.** `outputSchema`가 `type: object`와 `additionalProperties: true`뿐이라면, 계약이 말하는 것은 "결과는 객체다"가 전부입니다. 필드 이름을 하나도 약속하지 않습니다. 실제 결과에는 `success`, `session_id`, `status`, `missing_fields`, `next_question` 같은 키가 있는데, 그 의미는 outputSchema가 아니라 Skill 2~5항이 설명합니다. 이 서버는 반환 타입을 `dict[str, Any]`로 선언했기 때문에 SDK가 이 느슨한 스키마를 만든 것입니다. 반환 타입을 `BaseModel`로 선언하면 필드가 outputSchema에 실립니다(이 노트북은 그 변형을 만들어 보지 않았습니다).

결과의 키 중 `success`가 있다는 점을 기억해 두세요. 4절에서 업무 오류를 가르는 열쇠가 이것입니다. 이 호출도 1회 관찰입니다. 같은 입력을 여러 번 보내도 `session_id`만 달라집니다.

## 2. ToolAnnotations: 힌트와 강제의 차이

**문제.** 도구 정의에는 `annotations`라는 칸이 있고 `readOnlyHint: true` 같은 값이 들어갑니다. 이 값이 "읽기만 하는 도구는 쓰기 권한 없이 실행한다" 같은 규칙을 **만들어 줄까요?** 아니면 그냥 설명일까요? 설치된 `mcp_types`의 `ToolAnnotations` 설명은 "all properties in `ToolAnnotations` are **hints**"이고, 신뢰할 수 없는 서버의 값으로 도구 사용을 결정하지 말라고 적습니다(`.venv/lib/python3.13/site-packages/mcp_types/_v2026_07_28/__init__.py:1080~1085`). 힌트라는 말을 믿기 전에 코드로 확인합니다.

먼저 이 서버가 일곱 도구에 붙인 힌트를 표로 봅니다.

In [ ]:
# [따라하기] 일곱 도구의 annotations를 한 표로 읽습니다. 모델 호출 0회.
HINT_NAMES = ["readOnlyHint", "destructiveHint", "idempotentHint", "openWorldHint"]

hint_rows = []
for tool in rich_tools:
    hints = tool.get("annotations", {})
    hint_rows.append({"이름": tool["name"], **{name: hints.get(name, "(없음)") for name in HINT_NAMES}})
show_table(hint_rows)

**관찰할 것.** `readOnlyHint`가 `true`인 도구는 `list_evidence`와 `read_evidence` 둘뿐입니다(`cap_lab/server/server.py:52`의 `READ_ONLY`). 이름이 "조회"처럼 보이는 `get_session`은 `false`입니다. 서버 주석이 이유를 적고 있습니다(`cap_lab/server/server.py:312`). `get_session`은 파일에만 있는 기록을 DB로 복원하고 근거 파일을 다시 내보내기 때문에 읽기 전용이 아닙니다. 이름이 아니라 **실제 효과**로 힌트를 붙여야 한다는 사례입니다.

`idempotentHint`도 눈여겨보세요. 값이 `false`인 도구는 `start_interview`뿐이고, 읽기 전용 두 도구는 이 칸을 아예 보내지 않았습니다(표의 `(없음)`). 타입 정의가 `destructiveHint`와 `idempotentHint`는 `readOnlyHint == false`일 때만 의미가 있다고 적기 때문입니다(`.venv/lib/python3.13/site-packages/mcp_types/_v2026_07_28/__init__.py:1096`). 호출마다 새 세션 ID를 만들기 때문에, 응답을 놓쳤다고 같은 호출을 다시 보내면 세션이 하나 더 생깁니다. 이 구분이 Part 4의 재시도 설계로 이어집니다.

그렇다면 이 힌트를 **누가 읽고 지켜 줄까요?** 설치된 라이브러리 소스를 직접 뒤져 봅니다.

In [ ]:
# [따라하기] 힌트 이름이 설치된 패키지의 어느 파일에 나오는지 찾습니다. 모델 호출 0회.
import deepagents, fastmcp, langchain, langchain_core, mcp, mcp_types

PACKAGES = {"mcp": mcp, "mcp_types": mcp_types, "fastmcp": fastmcp,
            "langchain": langchain, "langchain_core": langchain_core, "deepagents": deepagents}
SEARCH_WORDS = ("read_only_hint", "readOnlyHint")

hint_files = {}
for package_name, package in PACKAGES.items():
    package_dir = Path(package.__file__).resolve().parent
    found = []
    for source in package_dir.rglob("*.py"):
        text = source.read_text(encoding="utf-8", errors="ignore")
        if any(word in text for word in SEARCH_WORDS):
            found.append(str(source.relative_to(package_dir)))
    hint_files[package_name] = found

for package_name, found in hint_files.items():
    print(f"{package_name}: {len(found)}개 파일", found[:4])

**관찰할 것.** `mcp`, `langchain`, `langchain_core`, `deepagents`에서는 이 이름이 나오는 파일이 0개입니다. `mcp_types`의 세 파일은 이름으로 보아 힌트 필드를 정의하는 타입 파일입니다(내용은 `.venv/lib/python3.13/site-packages/mcp_types/_v2026_07_28/__init__.py:1076~1120` 부분만 읽었습니다). `fastmcp`의 두 파일은 직접 열어 확인했습니다. `.venv/lib/python3.13/site-packages/fastmcp/_compat.py:46`은 `readOnlyHint`를 `read_only_hint`로 바꾸는 이름 대응표이고, `.venv/lib/python3.13/site-packages/fastmcp/server/transforms/resources_as_tools.py:36`은 자기가 만드는 도구에 `read_only_hint=True`를 붙이는 기본값입니다. 어느 것도 힌트를 읽어 호출을 막는 코드가 아닙니다. 다만 이 검색은 문자열 일치라서, 힌트를 읽어 동작을 바꾸는 코드가 이름을 다르게 쓰면 놓칩니다. 그래서 이 결과로 말할 수 있는 것은 "이 이름으로 힌트를 읽어 호출을 막는 코드를 찾지 못했다"까지입니다. 어떤 클라이언트(예: 사람에게 승인을 묻는 UI)는 이 힌트를 읽어 동작할 수 있습니다. 이 저장소의 Agent에서는 힌트가 권한이나 승인 규칙을 만들지 않는다고 읽어야 안전합니다.

**실제로 막는 것은 코드입니다.** 다음 셀은 서버에 장애 계획 `FaultPlan(permission_denied=["save_evidence"])`을 주고 같은 호출을 보냅니다. 거절은 `cap_lab/server/server.py:82`의 `fault_middleware`가 도구 코드에 닿기 전에 합니다. 이 서버는 annotations에 권한 칸이 없고, 거절 근거는 전적으로 미들웨어 코드입니다.

In [ ]:
# [따라하기] 권한 거부는 annotations가 아니라 서버 미들웨어 코드가 만듭니다. 모델 호출 0회.
def result_shape(result):
    """tools/call 결과가 어느 층의 결과인지 한 줄 요약으로 가른다.

    isError가 참이면 입력 검증·프로토콜 층, isError가 거짓이어도 structuredContent의 success가 거짓이면 업무 오류다.
    """
    structured = result.structured_content or {}
    text = result.content[0].text if result.content else ""
    if result.is_error:
        layer = "isError (검증·프로토콜 층)"
    elif structured.get("success") is False:
        layer = "업무 오류 (isError 아님, success=false)"
    else:
        layer = "성공"
    pydantic_type = re.search(r"\[type=(\w+)", text)
    return {
        "층": layer,
        "isError": result.is_error,
        "structuredContent": bool(structured),
        "error 코드": structured.get("error", ""),
        "검증 종류": pydantic_type.group(1) if pydantic_type else "",
        "본문 첫 줄": (text.splitlines() or [""])[0][:70],
    }


workdir_denied = new_workdir("denied")
async with connect_server(workdir_denied, fault=FaultPlan(permission_denied=["save_evidence"])) as client:
    started = await client.call_tool("start_interview", {"goal": "권한 거부를 보는 목표"})
    denied_session = started.structured_content["session_id"]
    denied = await client.call_tool("save_evidence", {"session_id": denied_session, "title": "메모", "content": "본문"})
    listed_after = await client.call_tool("list_evidence", {"session_id": denied_session})

show_table([
    {"호출": "save_evidence (거부 대상)", **result_shape(denied)},
    {"호출": "list_evidence (거부 대상 아님)", **result_shape(listed_after)},
])

거절된 호출이 `isError`가 아니라 **업무 오류**(`success=false`, `error: permission_denied`)로 돌아온 것을 보세요. 서버가 일부러 모델이 읽고 대응할 수 있는 모양으로 돌려준 것입니다. Skill 8항은 `permission_denied`를 받으면 같은 호출을 반복하지 말고 사용자에게 알리라고 적고 있습니다. 이 호출도 1회 관찰입니다. 힌트가 아니라 코드가 막는다는 점만 이 셀에서 가져가세요.

## 3. 서버가 발급하는 핸들

**문제.** `record_answer`는 `session_id`를 인자로 받습니다. 이 값은 누가 정할까요? 모델이 지어내면 안 되고, 사용자가 줄 수도 없습니다. 서버가 `start_interview` 결과에 실어 주고, 모델이 그 값을 **다음 도구 호출의 인자로 되돌려 보냅니다.** 2026-07-28 명세에서는 연결 단위 세션(`Mcp-Session-Id`)이 사라지고 이런 서버 발급 핸들을 도구 인자로 쓰는 방식이 권장됩니다(SEP-2567, `docs/DESIGN.md` 5절). 이 서버의 `session_id`가 바로 그 모양입니다. 발급은 `cap_lab/server/workflow.py:175`의 `uuid4().hex`입니다.

In [ ]:
# [따라하기] start_interview가 session_id를 발급하고, 같은 값이 다른 도구의 인자로 돌아옵니다. 모델 호출 0회.
workdir_handle = new_workdir("handle")
async with connect_server(workdir_handle) as client:
    started = await client.call_tool("start_interview", {"goal": "팀 비품 대여 현황을 보여 주는 도구"})
    issued_id = started.structured_content["session_id"]
    print("호출자가 넣은 인자: goal 하나뿐")
    print("서버가 발급한 session_id:", issued_id, f"(길이 {len(issued_id)})")

    fetched = await client.call_tool("get_session", {"session_id": issued_id})
    print("같은 id로 get_session ->", fetched.structured_content["status"], fetched.structured_content["missing_fields"])

    # next_question.field는 다음에 record_answer의 field 인자로 되돌려 보낼 값을 서버가 알려 주는 것이다.
    next_field = fetched.structured_content["next_question"]["field"]
    recorded = await client.call_tool(
        "record_answer", {"session_id": issued_id, "field": next_field, "answer": "없음"}
    )
    print("record_answer ->", recorded.structured_content["status"], "| 남은 질문:", recorded.structured_content["missing_fields"])

show_events(workdir_handle / "evidence")

**관찰할 것.** 세션 ID가 32자리 16진수라는 것, 그리고 `show_events`의 `세션` 칸이 그 앞 8자리라는 것을 맞춰 보세요(`cap_lab/notebook.py:98`). `next_question.field`도 같은 종류의 핸들입니다. 서버가 "다음에 이 값을 field로 보내라"고 알려 주고, 모델이 그대로 되돌려 보냅니다. 모델이 field 이름을 외우지 않아도 되도록 응답이 다음 인자를 안내하는 것입니다. 이 구조가 7절에서 plain의 약점을 일부 가려 줄 수 있는지 예측해 보세요.

scorer도 이 핸들을 씁니다. `args_valid`는 호출 인자의 `session_id`가 근거 폴더에 서버가 실제로 만든 세션 목록에 있는지 확인합니다(`cap_lab/scorers.py:169`). 서버가 만들지 않은 값을 모델이 지어내면 실패입니다. 이제 그 경우를 직접 만들어 봅니다.

In [ ]:
# [깨뜨리기] 서버가 발급하지 않은 session_id를 모양만 맞춰 넣어 봅니다. 모델 호출 0회.
FORGED_ID = "0" * 32   # 길이와 문자 종류는 맞지만 서버가 만든 적 없는 값

workdir_forged = new_workdir("forged")
async with connect_server(workdir_forged) as client:
    forged_calls = {
        "get_session (없는 id)": ("get_session", {"session_id": FORGED_ID}),
        "record_answer (없는 id)": ("record_answer", {"session_id": FORGED_ID, "field": "constraints", "answer": "없음"}),
        "get_session (빈 id)": ("get_session", {"session_id": ""}),
    }
    forged_rows = []
    for label, (tool_name, arguments) in forged_calls.items():
        result = await client.call_tool(tool_name, arguments)
        forged_rows.append({"호출": label, **result_shape(result)})
show_table(forged_rows)

**관찰할 것.** 모양이 맞아도 서버가 만든 적 없는 ID는 전부 같은 방식으로 거절됩니다. 거절은 `isError`가 아니라 **업무 오류**입니다. 서버가 입력의 형식(문자열인가)만 보고 통과시킨 뒤, 업무 코드가 "그런 세션이 없다"고 판단한 것이기 때문입니다. `error` 코드를 읽으세요. 모델이 이 코드를 보고 "ID를 잘못 썼으니 `start_interview`부터"라고 대응하려면 그 대응이 Skill에 적혀 있거나 오류 `message`가 알려 줘야 합니다. 빈 문자열 ID도 형식 검증은 통과합니다. 형식 검증은 "빈 문자열 금지"를 모르기 때문입니다.

이 절에서 스스로 확인할 것이 있습니다. 위에서 "모델이 지어낸 ID"를 막은 것은 어느 층이었나요? 열거형(`enum`)처럼 스키마가 막을 수 있는 값과, 서버의 상태를 알아야만 판정할 수 있는 값은 어떻게 다른가요? 다음 절이 그 구분을 정리합니다.

## 4. 업무 오류와 `isError`

**문제.** 호출이 실패하는 길은 두 가지입니다. 하나는 입력이 도구의 인자 형식(스키마)을 어긴 경우이고, 다른 하나는 형식은 맞지만 업무 규칙이 거절한 경우입니다. 모델과 scorer가 이 둘을 같은 방식으로 다루면 안 됩니다. 서버 코드에서 둘이 갈라지는 지점은 `cap_lab/server/server.py:202`의 `invoke`입니다. 업무 예외(`WorkflowError`, `EvidenceError`)만 잡아 결과 dict로 바꾸고, 그 밖의 실패는 SDK의 도구 오류로 넘깁니다. 입력 검증 실패는 도구 함수가 실행되기도 전에 SDK가 만듭니다.

**먼저 예측하세요.** `record_answer`에 아래 다섯 가지 호출을 보낼 때 각각 `isError`가 참일까요? `structuredContent`가 있을까요? 표를 직접 그려 보세요. (a) 정상 호출, (b) `field`에 허용값 밖의 `"color"`, (c) `answer`에 문자열이 아닌 숫자 `123`, (d) `answer`를 아예 빼먹음, (e) 서버가 발급한 적 없는 `session_id`.

In [ ]:
# [따라하기] 같은 도구에 다섯 가지 호출을 보내 결과 모양을 한 표로 가릅니다. 모델 호출 0회.
workdir_shapes = new_workdir("shapes")
async with connect_server(workdir_shapes) as client:
    started = await client.call_tool("start_interview", {"goal": "오류 모양을 비교하는 목표"})
    good_id = started.structured_content["session_id"]

    five_calls = {
        "(a) 정상": {"session_id": good_id, "field": "constraints", "answer": "없음"},
        "(b) field가 허용값 밖": {"session_id": good_id, "field": "color", "answer": "빨강"},
        "(c) answer가 숫자": {"session_id": good_id, "field": "constraints", "answer": 123},
        "(d) answer 누락": {"session_id": good_id, "field": "constraints"},
        "(e) 발급한 적 없는 id": {"session_id": FORGED_ID, "field": "constraints", "answer": "없음"},
    }
    shape_rows = []
    for label, arguments in five_calls.items():
        result = await client.call_tool("record_answer", arguments)
        shape_rows.append({"호출": label, **result_shape(result)})
show_table(shape_rows)

**관찰할 것.** 예측한 표와 맞춰 보세요. 읽는 법은 세 칸입니다.

- `isError`가 참인 행((b), (c), (d))은 **입력 검증 실패**입니다. 본문은 `Error executing tool record_answer: 1 validation error for record_answerArguments`로 시작하고, `검증 종류` 칸에 `literal_error`(열거형 위반), `string_type`(타입 위반), `missing`(필수 인자 누락)이 나옵니다. `structuredContent`가 없고 `error` 코드도 없습니다. 모델이 읽을 수 있는 것은 영어로 된 pydantic 메시지 본문뿐입니다.
- `isError`가 거짓인데 `success=false`인 행((e))은 **업무 오류**입니다. `structuredContent`에 `error` 코드와 한국어 `message`가 있어서 모델과 Skill이 코드로 분기할 수 있습니다.
- (b)에서 서버가 stderr에 `Tool 'record_answer' rejected arguments: ['field']`를 남긴 것도 이 층입니다(노트북 출력 칸에는 안 보이고 노트북을 띄운 터미널에 보일 수 있습니다). 도구 함수는 호출되지 않았으므로 DB에 아무것도 남지 않았습니다.

이 구분은 scorer에 그대로 반영됩니다. 실행 기록에서 `ok=False`는 두 경우 모두 참이지만, `args_valid`는 **검증 오류 문구가 있고 결과가 dict가 아닌 호출만** 인자 실패로 셉니다(`cap_lab/scorers.py:152`의 `args_valid`, 호출 기록 쪽은 `cap_lab/experiment.py:271`의 `_record_tool_message`). 업무 오류는 인자 형식 문제가 아니라 업무 흐름 문제이기 때문입니다. 이 정의는 7절의 plain 해석에서 중요해집니다. 이 다섯 호출도 각각 1회 관찰입니다.

In [ ]:
# [깨뜨리기] 존재하지 않는 도구 이름을 부르면 어떤 층의 어떤 결과가 올까요? 모델 호출 0회.
workdir_unknown = new_workdir("unknown-tool")
async with connect_server(workdir_unknown) as client:
    misspelled = await client.call_tool("record_answers", {"session_id": FORGED_ID})
    exact = await client.call_tool("record_answer", {"session_id": FORGED_ID})

show_table([
    {"호출": "record_answers (오타 s)", **result_shape(misspelled)},
    {"호출": "record_answer (인자 부족)", **result_shape(exact)},
])

**관찰할 것.** 도구 이름 오타는 인자 검증보다 앞에서 걸러집니다. `isError`가 참이고 본문은 "Unknown tool"입니다. `검증 종류` 칸이 비어 있는 것은 pydantic 검증까지 가지 못했다는 뜻입니다. 다만 이 노트북의 Agent에서는 모델이 목록에 없는 이름을 만들어 부르는 일을 프레임워크가 먼저 걸러 줄 수 있으므로, 이 결과를 "모델이 이름을 틀리면 이렇게 된다"로 일반화하지는 마세요. 그 경로는 이 노트북에서 확인하지 않았습니다.

## 5. rich 대 plain: 오프라인 비교

**문제.** 지금까지 본 서버는 도구 정의를 풍부하게 공개합니다(rich). 열거형, 설명, annotations, outputSchema가 모두 있습니다. 그렇다면 이것들이 **정말 모델의 행동을 바꿀까요?** 정의를 빼도 Skill과 서버 응답이 안내해 주므로 차이가 없을 수도 있습니다. 알려면 같은 서버를 정의만 다르게 띄우고 같은 사례를 돌려 보는 수밖에 없습니다.

서버의 `--interface plain` 옵션이 그 비교를 위해 있습니다(`cap_lab/server/server.py:116`의 `without_output_schema`와 `:193`). plain은 **같은 일곱 도구, 같은 인자 이름, 같은 업무 규칙**을 공개하되 아래를 뺍니다.

- `record_answer`의 `field` 열거형: 허용값을 모르는 평범한 문자열(`str`)이 됩니다.
- 도구 설명: 한 문장짜리 짧은 구절(예: "답변을 기록한다.")이 됩니다.
- ToolAnnotations와 outputSchema: 공개하지 않습니다.

SKILL.md는 plain에서도 그대로입니다. 그러므로 이 비교가 재는 것은 "안내가 아무것도 없을 때"가 아니라 **"도구 정의에 안내가 없고 Skill에만 있을 때"** 입니다. 결과를 해석할 때 이 점을 잊지 마세요. 먼저 두 정의를 오프라인으로 나란히 봅니다.

In [ ]:
# [따라하기] plain의 tools/list를 읽고 rich와 도구별로 나란히 비교합니다. 모델 호출 0회.
plain_tools = await fetch_tools("plain", "list-plain")
plain_by_name = {tool["name"]: tool for tool in plain_tools}

side_rows = []
for rich_tool in rich_tools:
    plain_tool = plain_by_name[rich_tool["name"]]
    side_rows.append({
        "이름": rich_tool["name"],
        "rich 설명": rich_tool["description"],
        "plain 설명": plain_tool["description"],
        "rich 설명 길이": len(rich_tool["description"]),
        "plain 설명 길이": len(plain_tool["description"]),
        "annotations (rich/plain)": f'{"annotations" in rich_tool}/{"annotations" in plain_tool}',
        "outputSchema (rich/plain)": f'{"outputSchema" in rich_tool}/{"outputSchema" in plain_tool}',
    })
show_table(side_rows)

rich_size = len(json.dumps(rich_tools, ensure_ascii=False))
plain_size = len(json.dumps(plain_tools, ensure_ascii=False))
print(f"tools/list 전체 글자 수: rich {rich_size} / plain {plain_size} (글자 수이고 토큰 수가 아닙니다)")

In [ ]:
# [따라하기] record_answer의 field 정의와 두 도구의 인자 이름·필수 목록이 같은지 확인합니다. 모델 호출 0회.
for name in ("record_answer",):
    rich_schema = next(tool for tool in rich_tools if tool["name"] == name)["inputSchema"]
    plain_schema = plain_by_name[name]["inputSchema"]
    print("rich  field:", rich_schema["properties"]["field"])
    print("plain field:", plain_schema["properties"]["field"])

same_arguments = all(
    set(tool["inputSchema"].get("properties", {})) == set(plain_by_name[tool["name"]]["inputSchema"].get("properties", {}))
    and tool["inputSchema"].get("required", []) == plain_by_name[tool["name"]]["inputSchema"].get("required", [])
    for tool in rich_tools
)
print("일곱 도구의 인자 이름과 필수 목록이 rich와 plain에서 같은가:", same_arguments)

In [ ]:
# [깨뜨리기] 같은 field 오타를 rich와 plain 서버에 보내 결과 모양을 비교합니다. 모델 호출 0회.
TYPO_FIELD = "constraint"   # constraints에서 s가 빠진 오타

typo_rows = []
for interface in ("rich", "plain"):
    workdir_typo = new_workdir(f"typo-{interface}")
    async with connect_server(workdir_typo, interface=interface) as client:
        started = await client.call_tool("start_interview", {"goal": "오타를 비교하는 목표"})
        typo_id = started.structured_content["session_id"]
        result = await client.call_tool("record_answer", {"session_id": typo_id, "field": TYPO_FIELD, "answer": "없음"})
        after = await client.call_tool("get_session", {"session_id": typo_id})
    typo_rows.append({
        "interface": interface, **result_shape(result),
        "기록된 답변 수": len(after.structured_content["answers"]),
    })
show_table(typo_rows)

**관찰할 것.** 같은 오타인데 두 서버의 반응 층이 다릅니다. rich는 열거형 위반이라 `isError`이고 `literal_error`입니다. plain은 `isError`가 아니라 **업무 오류 `unknown_field`** 로 돌아옵니다. 서버는 오타를 받아들여 업무 코드까지 호출을 보낸 뒤 거절한 것입니다(`cap_lab/server/workflow.py`의 `record_answer`가 `QUESTIONS`에 없는 field를 `unknown_field`로 거절). 두 경우 모두 기록된 답변은 0개입니다. 데이터는 안전하지만, **모델이 받는 오류 메시지의 질**이 다릅니다. rich는 "`'constraints' or 'acceptance_criteria'` 중 하나여야 한다"고 허용값을 알려 줍니다. plain의 메시지는 "응답의 next_question.field를 확인하세요"입니다.

여기서 6절에서 쓸 중요한 사실이 하나 나옵니다. plain에서 field 오타는 `isError`가 아니므로, 앞서 읽은 `args_valid`의 정의에 따르면 **인자 실패로 세지지 않습니다.** rich에서는 같은 실수가 `args_valid` 실패가 됩니다. 즉 두 인터페이스의 `args_valid` 숫자는 같은 잣대로 같은 실수를 세지 않습니다. 이것을 알고 결과를 읽어야 합니다.

## 6. 수업 중 실측: rich 대 plain 12회

**문제.** 풍부한 정의가 Claude의 인자 채우기와 도구 선택을 실제로 개선할까요?

**실험 설계.** 변형 두 개(`rich`, `plain`)는 `Variant`의 `interface`만 다릅니다(`cap_lab/experiment.py:79`). 게이트, 임계값, Skill 폴더, 도구 공개 방식은 모두 기본값으로 같습니다. 사례는 dev 사례 두 개를 골랐습니다. 실행은 2 사례 x 3 반복 x 2 변형 = 12회입니다.

**사례를 고른 이유.** 인터페이스의 차이는 **인자를 채우는 단계**에서 드러납니다. 그래서 결과가 인자에 달린 사례를 골랐습니다.

- `dev-03`(`goal_verbatim`): 사용자가 준 문장을 한 글자도 바꾸지 말고 `start_interview`의 `goal` 인자에 넣어야 합니다. 한 번의 호출에 인자 정확성이 걸립니다.
- `dev-07`(`full_flow`): 제약과 완료 기준을 `record_answer`로 각각 기록하고, `freeze_seed`로 확정하고, `save_evidence`로 저장해야 합니다. `field` 열거형을 두 번 채우고, `session_id`를 네 번 되돌려 보내고, 인자가 네 개인 `save_evidence`까지 갑니다. 도구 선택(`must_call` 네 개)과 인자가 모두 걸립니다.

제외한 사례도 이유가 있습니다. `dev-01`, `dev-02`는 Skill을 읽을지가 핵심이라 인터페이스와 상관이 적고, `dev-09`, `dev-10`은 장애 주입에 대한 재시도 행동을 보는 사례입니다. `dev-05`, `dev-06`은 완료 기준의 판정(H1 게이트)을 보는 사례입니다. 이 사례들로도 인터페이스를 비교할 수는 있지만, 인터페이스가 아닌 다른 요인이 결과를 지배해서 이 질문에는 깨끗한 증거가 못 됩니다.

**정직한 경고.** 변형마다 6번의 실행입니다. 사례 두 개의 3회 반복입니다. 이 숫자로 말할 수 있는 것은 매우 적습니다. 표의 Wilson 구간이 그 적음을 숫자로 보여 줍니다. "rich가 낫다"도 "차이가 없다"도 이 실행으로는 증명되지 않습니다.

**먼저 예측하세요.** 실행하기 전에 다섯 가지를 적으세요. (1) `tool_selection_ok`는 두 변형에서 차이가 날까요? 도구 이름과 인자 이름은 같다는 점을 떠올리세요. (2) `args_valid`는 어느 쪽이 높을까요? (3) plain에서 `field`를 틀리면 5절에서 본 대로 `args_valid`는 어떻게 될까요? (4) `dev-03`과 `dev-07` 중 어느 사례에서 차이가 날 가능성이 클까요? (5) Skill과 `next_question.field` 안내가 있다면, 열거형이 없어도 모델이 `field`를 틀리지 않을까요?

In [ ]:
# [따라하기] 유료: rich와 plain을 dev-03, dev-07로 3회씩 실행합니다.
announce("Part 2 비교: rich 대 plain", model_calls=110, runs=12)

rich_variant = Variant("rich", interface="rich")
plain_variant = Variant("plain", interface="plain")

compare_rows = run_small_compare(
    [rich_variant, plain_variant],
    ["dev-03", "dev-07"],
    3,
    label="part2-interface",
)
print("행 수:", len(compare_rows), "| 오류 행 수:", sum(1 for row in compare_rows if row["error"]))

In [ ]:
# [따라하기] 행마다 무엇이 일어났는지 그대로 봅니다. 오류가 난 행도 지우지 않습니다. 모델 호출 0회.
show_table(compare_rows, columns=["variant", "repeat", "case_id", "tool_selection_ok", "args_valid", "first_failure", "error"])

In [ ]:
# [따라하기] 변형과 검사별 통과 수와 Wilson 95% 구간을 계산합니다. 모델 호출 0회.
rate_table = pass_rate_table(compare_rows, ["args_valid", "tool_selection_ok"])
show_table(rate_table)

# 6번 모두 통과해도 구간의 아래쪽이 얼마나 낮은지 따로 봅니다.
low, high = wilson_ci(6, 6)
print(f"6/6 통과여도 Wilson 95% 구간은 {low:.2f}~{high:.2f}")
low, high = wilson_ci(3, 6)
print(f"3/6 통과의 구간은 {low:.2f}~{high:.2f}")

**관찰할 것.** 표를 읽는 순서를 지키세요. 먼저 `unchecked`(검사 못 함)와 오류 행 수를 보세요. 키가 없거나 호출이 실패해 `checked`가 0이면 `rate`가 `None`입니다. 확인하지 못한 것을 0%나 100%로 읽으면 안 됩니다. 다음으로 `passed`/`checked`와 구간을 보세요.

그리고 구간을 겹쳐 보세요. 두 변형의 구간이 크게 겹치면 이 실행은 "다르다"는 증거가 못 됩니다. 변형마다 6번뿐이라면 `6/6`이어도 구간의 아래쪽이 0.6 근처까지 내려갑니다. 위 셀의 마지막 두 줄이 그 숫자입니다. 게다가 구간은 6번이 **서로 독립**이라고 가정합니다. 이 6번은 사례 두 개의 반복이라 독립이 아닙니다. 같은 사례의 반복은 그 사례의 난이도를 공유하므로 진짜 불확실성은 이 구간보다 큽니다. 그래서 이 구간은 "얼마나 적게 증명하는지"의 하한입니다.

`first_failure` 칸은 한 실행에서 가장 먼저 실패한 단계(S1~S6)입니다. 같은 실패가 `dev-03`에서만 반복되는지, `dev-07`에서만 반복되는지 보세요. 그것이 위 예측 (4)의 답입니다. 이 행들은 반복 3회 중 3회가 모인 결과이므로 "1회 관찰"이 아닙니다. 그래도 사례 두 개라는 한계는 남습니다.

In [ ]:
# [따라하기] args_valid가 놓치는 것을 서버 쪽 기록으로 확인합니다. 모델 호출 0회.
def count_server_calls(rows):
    """실행마다 근거 폴더의 call-*.json(서버가 받은 모든 도구 호출)을 읽어 변형별로 센다.

    isError는 입력 검증 실패, 업무 오류는 isError가 아닌데 success=false인 결과다.
    """
    counts = {}
    for row in rows:
        entry = counts.setdefault(row["variant"], {
            "variant": row["variant"], "서버가 받은 호출": 0, "isError": 0, "업무 오류": 0,
        })
        for path in Path(row["evidence_dir"]).glob("*/call-*.json"):
            result = json.loads(path.read_text(encoding="utf-8"))["content"]["result"]
            entry["서버가 받은 호출"] += 1
            if result.get("isError"):
                entry["isError"] += 1
            elif (result.get("structuredContent") or {}).get("success") is False:
                entry["업무 오류"] += 1
    return list(counts.values())


show_table(count_server_calls(compare_rows))

In [ ]:
# [따라하기] 정의를 줄이면 입력 토큰이 줄어드는지, 행마다 기록된 수치로 봅니다. 모델 호출 0회.
def mean_cost(rows):
    """변형별로 오류 없는 실행의 평균 모델 호출·도구 호출·입력 토큰·출력 토큰·시간(초)을 계산한다.

    오류가 난 실행은 중간에 멈춰 수치가 불완전하므로 평균에서 뺀다.
    """
    table = []
    for variant in dict.fromkeys(row["variant"] for row in rows):
        clean = [row for row in rows if row["variant"] == variant and not row["error"]]
        entry = {"variant": variant, "오류 없는 실행 수": len(clean)}
        for name in ("model_calls", "tool_calls", "input_tokens", "output_tokens", "latency_s"):
            entry[f"평균 {name}"] = round(sum(row[name] for row in clean) / len(clean), 1) if clean else None
        table.append(entry)
    return table


show_table(mean_cost(compare_rows))

**관찰할 것.** `업무 오류` 칸은 `args_valid`가 보지 않는 실패입니다. plain에서 이 칸이 rich보다 크다면, 그 일부는 5절에서 본 `unknown_field`처럼 **인자 실수가 업무 오류로 둔갑한 것**일 수 있습니다. 반대로 두 변형의 `isError`와 `업무 오류`가 모두 0이거나 같다면, Claude가 Skill과 `next_question.field` 안내만으로도 필드를 맞게 채웠다는 뜻이고, 풍부한 정의가 이 사례에서 일을 더 하지 않았다는 증거입니다. 어느 쪽이든 이 표에는 어느 호출이 어느 사례에서 났는지가 없습니다. 더 파려면 `compare_rows`의 `evidence_dir` 아래 `call-*.json`을 직접 열어 보세요(경로를 출력해 보는 것으로 충분합니다).

비용 칸도 읽으세요. 바로 위 표의 `평균 input_tokens`는 호출마다 다시 보내는 입력의 크기라서, 도구 정의가 길수록 커질 수 있는 칸입니다. 다만 입력 토큰은 실행이 더 많은 단계를 거치거나 대화가 길어져도 같이 커지므로, 정의 길이의 효과만 따로 읽을 수는 없습니다. 이 평균은 오류 없는 실행만 모은 것이고, 이 표도 사례 두 개의 반복입니다. 정의를 줄여 아낀 토큰과 정확성의 손해를 맞바꾸는 판단은 이 12회로는 할 수 없습니다.

결론을 한 줄로 쓴다면 이렇습니다. **이 12회로는 "정의를 풍부하게 하라"와 "필요 없다" 어느 쪽도 증명되지 않는다. 증명되는 것은 두 인터페이스의 결과 모양이 다르고, `args_valid`가 그 둘을 같은 잣대로 세지 않는다는 것이다.**

## 직접 해보기

네 개 모두 모델을 부르지 않거나 잠겨 있습니다. 어느 것도 시험이 아니고 점수가 없습니다. 자기 말로 설명할 수 있는지가 기준입니다.

### 해보기 1. plain에서 사라진 정보를 직접 분류하기

plain이 뺀 정보를 하나씩 놓고, 그것이 사라지면 **무엇이 달라지는지** 적습니다. 처음 값은 이 노트북 저자의 추측입니다. 실측이나 읽은 코드와 다르면 고쳐 쓰세요. 값은 `인자 검증`, `도구 선택`, `오류 이해`, `영향 없음` 중 하나로 씁니다.

In [ ]:
# [내 것 만들기] 사라진 정보가 무엇에 영향을 주는지 분류합니다. 처음 값은 추측이니 고쳐 쓰세요. 모델 호출 0회.
LOST_INFORMATION = {
    "field 열거형": "인자 검증",
    "도구 설명 (긴 것 -> 짧은 구절)": "도구 선택",
    "ToolAnnotations": "영향 없음",
    "outputSchema": "영향 없음",
}
CHOICES = {"인자 검증", "도구 선택", "오류 이해", "영향 없음"}

for item, choice in LOST_INFORMATION.items():
    assert choice in CHOICES, f"{item}: 선택지 밖의 값입니다 -> {choice}"
show_table([{"사라진 정보": item, "내 분류": choice} for item, choice in LOST_INFORMATION.items()])

분류를 바꾼 이유를 한 문장으로 적어 보세요. 예를 들어 "ToolAnnotations는 2절에서 어떤 코드도 읽지 않았으므로 영향 없음"이라고 쓸 수 있습니다. 반대로 "열거형이 없으면 오류 이해가 달라진다"를 주장하려면 5절의 어느 칸을 근거로 들 수 있을까요?

### 해보기 2. 오타 후보를 바꿔 결과 층 알아보기

`FIELD_CANDIDATES`에 후보를 추가해 rich와 plain의 반응 층을 비교하세요. 공백, 대소문자, 하이픈, 한국어 같은 변형이 어느 층에서 걸리는지 예측하고 실행합니다.

In [ ]:
# [바꿔보기] FIELD_CANDIDATES를 바꿔 어떤 오타가 어느 층에서 걸리는지 봅니다. 모델 호출 0회.
FIELD_CANDIDATES = ["Constraints", " constraints", "acceptance-criteria", "제약", ""]

candidate_rows = []
for interface in ("rich", "plain"):
    workdir_candidates = new_workdir(f"candidates-{interface}")
    async with connect_server(workdir_candidates, interface=interface) as client:
        started = await client.call_tool("start_interview", {"goal": "후보를 비교하는 목표"})
        candidate_id = started.structured_content["session_id"]
        for candidate in FIELD_CANDIDATES:
            result = await client.call_tool(
                "record_answer", {"session_id": candidate_id, "field": candidate, "answer": "없음"}
            )
            shape = result_shape(result)
            candidate_rows.append({"interface": interface, "field": repr(candidate), "층": shape["층"],
                                   "error 코드": shape["error 코드"], "검증 종류": shape["검증 종류"]})
show_table(candidate_rows)

공백이 앞에 붙은 `" constraints"`처럼 사람 눈에는 같아 보이는 값이 두 인터페이스에서 같은 층에서 걸리는지 확인하세요. 서버가 입력을 다듬어(`strip`) 받아 주는지는 `cap_lab/server/workflow.py`의 `record_answer`가 `field`를 `QUESTIONS`에서 그대로 찾는지 읽어 보면 알 수 있습니다. 다듬어 받아 주는 서버는 친절하지만 모델이 틀린 값을 보냈다는 사실을 가립니다. 어느 쪽이 나은지 의견을 적어 보세요.

### 해보기 3. 다른 사례 쌍으로 1회 관찰

같은 비교를 다른 두 사례로 **1회만** 돌려 봅니다. 6절과 같은 인터페이스 비교를 다른 사례에서도 보고 싶을 때입니다. 반복이 1회이므로 결과는 1회 관찰이고, 6절 결과와 합쳐 통과율을 계산하지 마세요. 서로 다른 사례 선택의 결과를 합치면 처음 계획에 없던 비교가 됩니다.

In [ ]:
# [바꿔보기] 유료: 다른 dev 사례 두 개를 rich/plain으로 한 번씩만 돌립니다. 잠겨 있으므로 켜려면 True로 바꾸세요.
RUN_EXERCISE = False
EXERCISE_CASES = ["dev-04", "dev-07"]   # 인자에 걸리는 다른 사례로 바꿔도 됩니다

if RUN_EXERCISE:
    announce("해보기 3: 다른 사례 1회 관찰", model_calls=40, runs=4)
    exercise_rows = run_small_compare(
        [Variant("rich", interface="rich"), Variant("plain", interface="plain")],
        EXERCISE_CASES,
        1,
        label="part2-exercise",
    )
    show_table(exercise_rows, columns=["variant", "repeat", "case_id", "tool_selection_ok", "args_valid", "first_failure", "error"])
else:
    print("RUN_EXERCISE = False 이므로 실행하지 않았습니다. 모델 호출 0회.")

### 해보기 4. 선택: 처음부터 5회로 정해 새로 돌리기

6절의 결과가 마음에 들지 않아 반복을 늘리는 것은 하지 마세요. 결과를 본 뒤 반복을 늘려 판정을 바꾸는 일은 이 과정의 본 측정이 사전 등록에서 금지한 행동입니다(`docs/DESIGN.md` P5 끝줄). 아래 셀은 6절과 이어 붙이는 것이 아니라 **처음부터 5회로 정한 별도 실험**입니다. 켜기 전에 "무엇이 나오면 어떻게 해석하겠다"를 먼저 적으세요. 그 해석은 결과를 보고 바꾸지 않습니다.

In [ ]:
# [바꿔보기] 유료, 선택: dev-03과 dev-07을 5회씩 새로 돌립니다. 잠겨 있으므로 켜려면 True로 바꾸세요.
RUN_OPTIONAL = False

if RUN_OPTIONAL:
    announce("해보기 4: 5회 반복 비교", model_calls=190, runs=20)
    optional_rows = run_small_compare(
        [Variant("rich", interface="rich"), Variant("plain", interface="plain")],
        ["dev-03", "dev-07"],
        5,
        label="part2-five-repeats",
    )
    show_table(pass_rate_table(optional_rows, ["args_valid", "tool_selection_ok"]))
else:
    print("RUN_OPTIONAL = False 이므로 실행하지 않았습니다. 모델 호출 0회.")

## 확인 질문

답을 말이나 글로 설명해 보세요. 점수는 없습니다. 막히는 질문이 있으면 해당 절의 셀을 다시 읽고 출력 칸을 짚어 설명해 보세요.

1. `record_answer`의 `field` 열거형은 서버 코드에서 JSON Schema의 어떤 키로 나가고, 열거형을 어긴 호출은 어느 층에서 어떤 모양으로 돌아옵니까? plain에서는 같은 실수가 어떻게 달라집니까?
2. `readOnlyHint: true`는 도구가 쓰기를 못 하게 막습니까? 이 노트북에서 어떤 셀이 그 답을 보여 줬고, 무엇이 실제로 호출을 막았습니까?
3. `session_id`는 누가 만들고, 모델이 지어낸 값이 들어오면 어느 층이 거절합니까? `args_valid`가 그 경우를 어떻게 센다고 읽었습니까?
4. 업무 오류와 `isError` 오류는 `structuredContent`와 `error` 코드 측면에서 어떻게 다르고, 모델에게 왜 이 차이가 중요합니까?
5. 6절의 12회 실행으로 "rich가 더 낫다"를 말할 수 있습니까? Wilson 구간과 반복의 독립성 문제를 들어 설명하세요.

## 정리

- 모델은 서버 코드를 보지 못하고 `tools/list`의 이름, 설명, `inputSchema`(열거형 포함), `outputSchema`, annotations만 봅니다. 이 서버의 `outputSchema`는 `dict` 반환이라 필드 계약을 주지 않았고, 필드의 의미는 Skill과 응답이 전했습니다.
- JSON Schema 2020-12는 `$schema`가 없을 때의 기본값입니다. 이 서버는 `$schema`를 보내지 않았습니다.
- ToolAnnotations는 힌트입니다. 설치된 패키지에서 힌트를 읽어 동작을 바꾸는 코드를 찾지 못했고, 실제 거절은 서버 미들웨어 코드(`permission_denied`)가 했습니다.
- 입력 검증 실패는 `isError`이고 `structuredContent`가 없으며, 업무 오류는 `isError`가 아니면서 `success=false`와 `error` 코드를 가집니다. scorer `args_valid`는 앞의 것만 인자 실패로 셉니다.
- `session_id`는 서버가 발급하고 모델이 되돌려 보내는 핸들입니다. 지어낸 값은 업무 오류로 거절됩니다.
- plain은 같은 일곱 도구와 같은 업무 규칙에서 열거형, 설명, annotations, outputSchema만 뺍니다. 같은 `field` 오타가 rich에서는 `isError`, plain에서는 업무 오류 `unknown_field`로 돌아와 `args_valid`가 다르게 셉니다.
- 12회 실행은 인터페이스의 우열을 증명하지 못합니다. 변형마다 6번이고 사례 두 개의 반복이라 구간은 넓고 독립도 아닙니다. 이 한계를 숫자로 말하는 것이 이 절의 목표였습니다.